# ESG 與 ROA 研究重做版（新資料）

本 notebook 參考前一版流程，但改用目前整理好的新資料來重做。

這份 notebook 會完成：
1. 載入研究資料
2. 檢查欄位與缺失值
3. 建立研究模型資料
4. EDA 與圖表
5. OLS / Linear Regression
6. 異質性分析（Large Firm）
7. 大 / 小公司分組回歸
8. Machine Learning（Random Forest / XGBoost）
9. 模型比較圖與報告用圖表

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import statsmodels.api as sm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

plt.rcParams["figure.dpi"] = 130
sns.set_theme(style="whitegrid")

### Step 1｜載入研究主表

這一步對應你舊 notebook 的「直接讀入整理後研究主表」。
如果你已經有 `master_full_merged.csv` 或 `model_base_merged.csv`，就從這裡開始。

這裡我建議：
- 做完整分析：用 `master_full_merged.csv`
- 做簡化 baseline：用 `model_base_merged.csv`

In [ ]:
df = pd.read_csv("master_full_merged.csv", encoding="utf-8-sig")
print("資料 shape:", df.shape)
display(df.head())
print(df.columns.tolist())

### Step 2｜檢查目前有哪些欄位

先確認：
- ESG / E / S / G 是否存在
- ROA 是否存在
- 市值與市佔是否存在
- 年度與公司代號格式是否正常

In [ ]:
print("欄位列表：")
for c in df.columns:
    print(c)

print("\nROA 相關欄位：", [c for c in df.columns if "ROA" in c])
print("市值相關欄位：", [c for c in df.columns if "市值" in c])
print("市佔相關欄位：", [c for c in df.columns if "市佔" in c])

### Step 3｜統一格式

先把公司代號與年度統一，方便後面建立 lag / next 變數。

In [ ]:
df["代號"] = df["代號"].astype(str).str.strip()
df["年度"] = pd.to_numeric(df["年度"], errors="coerce")

df = df.sort_values(["代號", "年度"]).copy()

print("公司數量:", df["代號"].nunique())
print("年份:", sorted(df["年度"].dropna().unique()))

### Step 4｜檢查缺失值比例

這一步對應舊 notebook 的資料品質檢查。
可以快速看出哪些欄位不能用，哪些欄位可以留在模型裡。舊版 notebook 也特別強調過，很多市場欄位可能是全空，因此正式模型要避開高缺失欄位。

In [ ]:
missing_ratio = df.isna().mean().sort_values(ascending=False)
display(missing_ratio.head(30))

### Step 5｜設定研究核心欄位

這一步參考你舊版分析 notebook 的做法：
- 目標變數：ROA
- 核心自變數：ESG、E、S、G
- 控制變數：公司規模

舊版 notebook 的 baseline 也先從 ESG、E、S、G 加公司規模開始。

In [ ]:
# 目標欄位
target_base = "ROA_avg"

# 先確認核心欄位
core_cols = [
    "代號", "名稱", "年度",
    "TESG分數", "環境構面分數", "社會構面分數", "公司治理構面分數",
    "市值_avg",
    "事務所_產業市佔率_營收 / 上市櫃與櫃營收 %",
    target_base
]

core_cols = [c for c in core_cols if c in df.columns]
research_df = df[core_cols].copy()

print("研究欄位：", core_cols)
display(research_df.head())

### Step 6｜建立前一期與下一期 ROA

這一步是新版升級版設計。
因為 ESG 對績效通常有時間差，所以除了當期 ROA，也建立：
- `ROA_lag`：前一年 ROA
- `ROA_next`：下一年 ROA

這比單純做當期 ESG 對當期 ROA 更有研究意義。

In [ ]:
research_df = research_df.sort_values(["代號", "年度"]).copy()

research_df["ROA_lag"] = research_df.groupby("代號")[target_base].shift(1)
research_df["ROA_next"] = research_df.groupby("代號")[target_base].shift(-1)

display(
    research_df[["代號", "名稱", "年度", "ROA_avg", "ROA_lag", "ROA_next"]].head(12)
)

### Step 7｜建立大公司 / 小公司變數

這一步直接仿照你舊版 notebook 的異質性分析邏輯：
用公司規模切成 Large Firm 與 Small Firm，再建立交互作用項。

舊版 notebook 是用規模中位數做切分，並建立 `ESG_x_Large`。

In [ ]:
size_col = "市值_avg" if "市值_avg" in research_df.columns else None

if size_col is not None:
    median_size = research_df[size_col].median()
    research_df["Large_Firm"] = (research_df[size_col] > median_size).astype(int)
    research_df["Small_Firm"] = (research_df[size_col] <= median_size).astype(int)
    research_df["ESG_x_Large"] = research_df["TESG分數"] * research_df["Large_Firm"]

display(
    research_df[["代號", "名稱", "年度", size_col, "Large_Firm", "Small_Firm", "TESG分數", "ESG_x_Large"]].head()
)

### Step 8｜描述統計

這一步對應舊 notebook 的 descriptive statistics。
這張表很適合直接放進報告。舊版也是先做主要變數的平均數、標準差、分位數與極值表。

In [ ]:
desc_cols = [
    "TESG分數", "環境構面分數", "社會構面分數", "公司治理構面分數",
    "市值_avg", "ROA_avg", "ROA_lag", "ROA_next"
]
desc_cols = [c for c in desc_cols if c in research_df.columns]

display(research_df[desc_cols].describe().T)

### Step 9｜主要分析欄位缺失值表

這張表可以放進 appendix 或方法章，說明哪些欄位可用。

In [ ]:
main_check_cols = [
    "TESG分數", "環境構面分數", "社會構面分數", "公司治理構面分數",
    "市值_avg", "ROA_avg", "ROA_lag", "ROA_next",
    "事務所_產業市佔率_營收 / 上市櫃與櫃營收 %"
]
main_check_cols = [c for c in main_check_cols if c in research_df.columns]

missing_main = research_df[main_check_cols].isna().mean().sort_values(ascending=False)
display(missing_main)


•	ROA_next = 0.165493
→ 約 16.5% 缺值

•	ROA_lag = 0.151408
→ 約 15.1% 缺值

•	ROA_avg = 0.024648
→ 約 2.5% 缺值

•	TESG分數 = 0.000000
→ 完全沒有缺值

•	市值_avg = 0.000000
→ 完全沒有缺值

畫一下敘述統計圖

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

# ===== 1. 指定要做描述統計的欄位 =====
desc_map = {
    "ROA_avg": "ROA",
    "TESG分數": "ESG總分",
    "環境構面分數": "E（環境）",
    "社會構面分數": "S（社會）",
    "公司治理構面分數": "G（治理）"
}

# 只保留實際存在的欄位
desc_cols = [c for c in desc_map.keys() if c in research_df.columns]

# ===== 2. 建立描述統計表 =====
rows = []

for col in desc_cols:
    s = pd.to_numeric(research_df[col], errors="coerce").dropna()

    rows.append({
        "變數": desc_map[col],
        "樣本數": int(s.count()),
        "均值": round(s.mean(), 4),
        "標準差": round(s.std(), 4),
        "最小值": round(s.min(), 4),
        "最大值": round(s.max(), 4),
        "偏度": round(skew(s, bias=False), 4),
        "峰度": round(kurtosis(s, fisher=False, bias=False), 4)  # 常見表格用 Pearson kurtosis
    })

desc_table = pd.DataFrame(rows)

# ===== 3. 顯示結果 =====
display(desc_table)

# ===== 4. 如果你要匯出成 CSV =====
desc_table.to_csv("descriptive_statistics_table.csv", index=False, encoding="utf-8-sig")
print("已輸出：descriptive_statistics_table.csv")

In [ ]:
desc_table_style = desc_table.copy()

for col in ["均值", "標準差", "最小值", "最大值", "偏度", "峰度"]:
    desc_table_style[col] = desc_table_style[col].map(lambda x: f"{x:.2f}")

display(desc_table_style)

In [ ]:
n_obs = research_df["ROA_avg"].notna().sum()
roa_mean = pd.to_numeric(research_df["ROA_avg"], errors="coerce").mean()
roa_std  = pd.to_numeric(research_df["ROA_avg"], errors="coerce").std()

esg_mean = pd.to_numeric(research_df["TESG分數"], errors="coerce").mean()
esg_min  = pd.to_numeric(research_df["TESG分數"], errors="coerce").min()
esg_max  = pd.to_numeric(research_df["TESG分數"], errors="coerce").max()

e_mean = pd.to_numeric(research_df["環境構面分數"], errors="coerce").mean()
s_mean = pd.to_numeric(research_df["社會構面分數"], errors="coerce").mean()
g_mean = pd.to_numeric(research_df["公司治理構面分數"], errors="coerce").mean()

print(f"樣本數：{n_obs}")
print(f"ROA均值：{roa_mean:.2f}（標準差：{roa_std:.2f}）")
print(f"ESG均值：{esg_mean:.2f}（範圍：{esg_min:.1f} - {esg_max:.1f}）")
print(f"E/S/G均值：{e_mean:.1f} / {s_mean:.1f} / {g_mean:.1f}")

### Step 10｜主要變數相關係數熱圖

這一步完全仿照舊版 notebook 的熱圖概念，但改用你現在的新欄位。舊版也是先用 ESG、E、S、G 與 ROA 畫相關係數熱圖。

In [ ]:
# 定義要用的欄位（一定要先跑這行🔥）
corr_cols = [
    "TESG分數",
    "環境構面分數",
    "社會構面分數",
    "公司治理構面分數",
    "ROA_avg",
    "ROA_lag",
    "ROA_next"
]

# 過濾存在的欄位（避免 KeyError）
corr_cols = [c for c in corr_cols if c in research_df.columns]
# 建立英文對照表
rename_map = {
    "TESG分數": "ESG Score",
    "環境構面分數": "Environmental Score",
    "社會構面分數": "Social Score",
    "公司治理構面分數": "Governance Score",
    "ROA_avg": "ROA",
    "ROA_lag": "ROA (t-1)",
    "ROA_next": "ROA (t+1)"
}

# 重新命名欄位（只用於畫圖）
corr_df = research_df[corr_cols].rename(columns=rename_map)

corr_mat = corr_df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_mat, annot=True, cmap="coolwarm", fmt=".2f")

plt.title("Correlation Matrix of ESG and ROA Variables")

sns.heatmap(
    corr_mat,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    center=0,          # 讓0變中間色（更好看）
    linewidths=0.5
)
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.savefig("correlation_heatmap.png")
plt.show()


### Step 11｜ESG / E / S / G 與 ROA 的散點圖

這一步對應舊版 notebook 的散點圖，但我幫你一次畫得更完整，方便放報告。

In [ ]:
# 英文名稱對照（畫圖用）
name_map = {
    "TESG分數": "ESG Score",
    "環境構面分數": "Environmental Score",
    "社會構面分數": "Social Score",
    "公司治理構面分數": "Governance Score",
    "ROA_avg": "ROA"
}

# 要畫的配對
plot_pairs = [
    ("TESG分數", "ROA_avg"),
    ("環境構面分數", "ROA_avg"),
    ("社會構面分數", "ROA_avg"),
    ("公司治理構面分數", "ROA_avg")
]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (xcol, ycol) in zip(axes.flatten(), plot_pairs):
    if xcol in research_df.columns and ycol in research_df.columns:

        sns.regplot(
            data=research_df,
            x=xcol,
            y=ycol,
            scatter_kws={"alpha": 0.5},
            line_kws={"color": "darkred", "linewidth": 2},
            ax=ax
        )

        ax.set_title(f"{name_map[xcol]} vs {name_map[ycol]}", fontsize=12)
        ax.set_xlabel(name_map[xcol])
        ax.set_ylabel(name_map[ycol])

plt.tight_layout()

# ⭐ 存圖（超重要）
plt.savefig("Fig2_ESG_vs_ROA_scatter.png", dpi=300, bbox_inches="tight")

plt.show()

### Step 12｜大公司與小公司的 ROA 分布比較

這張圖是新加的，報告很好用。
可以直觀看大公司與小公司的績效分布差異。

In [ ]:
if "Large_Firm" in research_df.columns:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=research_df, x="Large_Firm", y="ROA_avg")
    plt.title("ROA Distribution: Small vs Large Firms")
    plt.xlabel("Large_Firm (0=Small, 1=Large)")
    plt.ylabel("ROA_avg")
    plt.savefig("bigsm_roabox.png", dpi=300, bbox_inches="tight")

    plt.show()

### Step 13｜建立 OLS / Linear Regression 用資料表

這一步對應舊版 notebook 的模型資料建立：
- 保留 target 與特徵
- 只刪 target 缺值
- 自變數用中位數補值

舊版也是這樣處理，以避免直接 dropna 讓樣本太少。

In [ ]:
target = "ROA_avg"

candidate_features = [
    "TESG分數",
    "環境構面分數",
    "社會構面分數",
    "公司治理構面分數",
    "市值_avg"
]

candidate_features = [c for c in candidate_features if c in research_df.columns]

model_df = research_df[[target] + candidate_features].copy()
model_df = model_df.dropna(subset=[target])

for col in candidate_features:
    med = model_df[col].median()
    if pd.notna(med):
        model_df[col] = model_df[col].fillna(med)

print("模型資料 shape:", model_df.shape)
display(model_df.head())

### Step 14｜Train / Test Split

這一步對應舊版 notebook 的 baseline 模型切分。

In [ ]:
X = model_df[candidate_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

### Step 15｜Linear Regression 基礎模型

這一步對應舊版 notebook 的 baseline linear model。

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Linear Regression")
print("R2  :", r2_score(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("MAE :", mean_absolute_error(y_test, y_pred_lr))

### Step 16｜線性回歸係數表

這一步也仿照舊版 notebook，方便解釋方向：
- 正係數：正向關係
- 負係數：負向關係

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_
}).sort_values("Coefficient", ascending=False)

display(coef_df)

### Step 17｜OLS 摘要表

這一步對應舊版 notebook 的 statsmodels OLS summary，
可以直接拿來看 p 值與顯著性。

In [ ]:
X_sm = sm.add_constant(X)
ols_model = sm.OLS(y, X_sm).fit()
print(ols_model.summary())

### Step 18｜異質性分析：ESG × Large Firm

這一步直接仿照你舊版 notebook 的 interaction model：
ROA = ESG + Large_Firm + ESG_x_Large + controls

舊版 notebook 就是用這個模型檢查 ESG 在大公司與小公司之間是否不同。

In [ ]:
features_het = [
    "TESG分數",
    "市值_avg",
    "Large_Firm",
    "ESG_x_Large"
]
features_het = [c for c in features_het if c in research_df.columns]

het_df = research_df[[target] + features_het].copy()
het_df = het_df.dropna(subset=[target])

for col in features_het:
    med = het_df[col].median()
    if pd.notna(med):
        het_df[col] = het_df[col].fillna(med)

X_het = sm.add_constant(het_df[features_het])
y_het = het_df[target]

model_het = sm.OLS(y_het, X_het).fit()
print(model_het.summary())

### Step 19｜大公司與小公司分組回歸

這一步也沿用舊版 notebook 的設計：
直接把樣本切成兩組，各自跑 OLS，比較 ESG 係數。

In [ ]:
large_df = research_df[research_df["Large_Firm"] == 1].copy()
small_df = research_df[research_df["Large_Firm"] == 0].copy()

features_group = [
    "TESG分數",
    "環境構面分數",
    "社會構面分數",
    "公司治理構面分數",
    "市值_avg"
]
features_group = [c for c in features_group if c in research_df.columns]

def run_group_reg(data, name):
    temp = data[[target] + features_group].copy()
    temp = temp.dropna(subset=[target])
    for col in features_group:
        med = temp[col].median()
        if pd.notna(med):
            temp[col] = temp[col].fillna(med)
    Xg = sm.add_constant(temp[features_group])
    yg = temp[target]
    model = sm.OLS(yg, Xg).fit()
    print("=" * 60)
    print(name)
    print(model.summary())
    return model

model_large = run_group_reg(large_df, "Large Firms")
model_small = run_group_reg(small_df, "Small Firms")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))

# 小公司（藍色）
sns.regplot(
    data=research_df[research_df["Large_Firm"] == 0],
    x="TESG分數",
    y="ROA_avg",
    scatter_kws={"alpha":0.4},
    line_kws={"color":"blue", "label":"Small Firms"}
)

# 大公司（紅色）
sns.regplot(
    data=research_df[research_df["Large_Firm"] == 1],
    x="TESG分數",
    y="ROA_avg",
    scatter_kws={"alpha":0.4},
    line_kws={"color":"red", "label":"Large Firms"}
)

# 標題與標籤（英文版）
plt.title("ESG vs ROA: Small vs Large Firms")
plt.xlabel("ESG Score")
plt.ylabel("ROA")

plt.legend()

# 存圖🔥
plt.savefig("Fig_ESG_Heterogeneity.png", dpi=300, bbox_inches="tight")

plt.show()

### Step 20｜Random Forest 非線性輔助模型

這一步對應舊版 notebook 的非線性輔助分析。

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest")
print("R2  :", r2_score(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("MAE :", mean_absolute_error(y_test, y_pred_rf))

### Step 21｜XGBoost 模型

這一步是新版補強版。
舊版以 RF 為主要 ML 輔助模型；你現在可以再多一個 XGBoost 做比較。

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost")
print("R2  :", r2_score(y_test, y_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
print("MAE :", mean_absolute_error(y_test, y_pred_xgb))

### Step 22｜Feature Importance

這一步延續舊版 notebook 的 feature importance 圖，但現在同時可看 RF / XGB。

In [ ]:
rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

# 英文對照表
name_map = {
    "TESG分數": "ESG Score",
    "環境構面分數": "Environmental Score",
    "社會構面分數": "Social Score",
    "公司治理構面分數": "Governance Score",
    "市值_avg": "Firm Size"
}

# 建立英文版 DataFrame（只用來畫圖）
rf_imp_plot = rf_importance.copy()
rf_imp_plot["Feature_EN"] = rf_imp_plot["Feature"].map(name_map)

# 如果有沒對應到的欄位，就保留原名
rf_imp_plot["Feature_EN"] = rf_imp_plot["Feature_EN"].fillna(rf_imp_plot["Feature"])

display(rf_imp_plot)

plt.figure(figsize=(8, 5))

rf_imp_plot.sort_values("Importance").plot(
    x="Feature_EN",
    y="Importance",
    kind="barh",
    legend=False
)

plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Features")

plt.tight_layout()

# ⭐ 存圖（報告用）
plt.savefig("Fig3_RF_Feature_Importance.png", dpi=300, bbox_inches="tight")

plt.show()

### Step 23｜Actual vs Predicted 圖

這一步直接延續舊版 notebook 的模型預測圖。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Linear
axes[0].scatter(y_test, y_pred_lr, alpha=0.7)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_title("Linear Regression")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")

# RF
axes[1].scatter(y_test, y_pred_rf, alpha=0.7)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[1].set_title("Random Forest")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")

# XGB
axes[2].scatter(y_test, y_pred_xgb, alpha=0.7)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[2].set_title("XGBoost")
axes[2].set_xlabel("Actual")
axes[2].set_ylabel("Predicted")

plt.tight_layout()
plt.savefig("vs.png", dpi=300, bbox_inches="tight")
plt.show()

### Step 24｜模型結果整理表

這一步對應舊版 notebook 的結果 summary table。

In [ ]:
results_summary = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "R2": r2_score(y_test, y_pred_lr),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        "MAE": mean_absolute_error(y_test, y_pred_lr)
    },
    {
        "Model": "Random Forest",
        "R2": r2_score(y_test, y_pred_rf),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        "MAE": mean_absolute_error(y_test, y_pred_rf)
    },
    {
        "Model": "XGBoost",
        "R2": r2_score(y_test, y_pred_xgb),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
        "MAE": mean_absolute_error(y_test, y_pred_xgb)
    }
])

display(results_summary.sort_values("R2", ascending=False))

In [ ]:
results_data = [
    {
        "Model": "Linear Regression",
        "R2": r2_score(y_test, y_pred_lr),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        "MAE": mean_absolute_error(y_test, y_pred_lr)
    },
    {
        "Model": "Random Forest",
        "R2": r2_score(y_test, y_pred_rf),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        "MAE": mean_absolute_error(y_test, y_pred_rf)
    },
    {
        "Model": "XGBoost",
        "R2": r2_score(y_test, y_pred_xgb),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
        "MAE": mean_absolute_error(y_test, y_pred_xgb)
    }
]

# Conditionally add LightGBM results
if 'y_pred_lgbm' in locals():
    results_data.append({
        "Model": "LightGBM",
        "R2": r2_score(y_test, y_pred_lgbm),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_lgbm)),
        "MAE": mean_absolute_error(y_test, y_pred_lgbm)
    })
else:
    print("Warning: y_pred_lgbm is not defined. LightGBM results will not be included.")

# Conditionally add SVM results
if 'y_pred_svm' in locals():
    results_data.append({
        "Model": "SVM",
        "R2": r2_score(y_test, y_pred_svm),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_svm)),
        "MAE": mean_absolute_error(y_test, y_pred_svm)
    })
else:
    print("Warning: y_pred_svm is not defined. SVM results will not be included.")

results_summary = pd.DataFrame(results_data)
display(results_summary.sort_values("R2", ascending=False))

In [ ]:
import joblib

# Re-define features here to ensure it's in scope for saving the model.
# This list should match the features used for training the ROA_next prediction models.
_temp_features_candidate = [
    "TESG分數",
    "環境構面分數",
    "社會構面分數",
    "公司治理構面分數",
    "市值_avg",
    "ROA_lag"
]

# Assuming 'research_df' is globally available from previous cells
# This step replicates the filtering logic from cell 'qDyNciutG_x5'
if 'research_df' in globals():
    features = [c for c in _temp_features_candidate if c in research_df.columns]
else:
    # Fallback if research_df is not defined. Using the full list for robustness.
    # In a real scenario, ensure research_df is always processed before saving.
    features = _temp_features_candidate
    print("Warning: 'research_df' not found. Using assumed features list for saving. Please ensure 'research_df' is defined and processed before running this cell.")


# 選擇你想要儲存的模型，這裡以 Random Forest 為例
best_model = rf # 這裡可以換成 lgbm 或 pipeline_svm 等

# 儲存模型
joblib.dump(best_model, 'roa_prediction_model.joblib')

# 儲存特徵列表
joblib.dump(features, 'model_features.joblib')

print("模型 'roa_prediction_model.joblib' 已儲存。")
print("特徵列表 'model_features.joblib' 已儲存。")

In [ ]:
import joblib

# 確保 features 變數已經定義（從之前的儲存模型 cell 複製過來，確保獨立性）
_temp_features_candidate = [
    "TESG分數",
    "環境構面分數",
    "社會構面分數",
    "公司治理構面分數",
    "市值_avg",
    "ROA_lag"
]

# 假設 'research_df' 是全局可用的
if 'research_df' in globals():
    features = [c for c in _temp_features_candidate if c in research_df.columns]
else:
    features = _temp_features_candidate
    print("Warning: 'research_df' not found. Using assumed features list for saving.")

# 選擇你想要儲存的模型，這裡以 Random Forest 為例
best_model_to_save = rf # 這裡可以換成 lgbm 或 pipeline_svm 等

# 儲存模型為 .pkl 檔案
joblib.dump(best_model_to_save, 'roa_prediction_model.pkl')

# 儲存特徵列表為 .pkl 檔案
joblib.dump(features, 'model_features.pkl')

print("模型 'roa_prediction_model.pkl' 已儲存。")
print("特徵列表 'model_features.pkl' 已儲存。")

儲存模型後，你可以在你的網頁應用程式中載入它，然後用它來預測。以下是載入模型和特徵，並進行預測的範例程式碼：

```python
import joblib
import pandas as pd

# 載入模型和特徵
loaded_model = joblib.load('roa_prediction_model.joblib')
loaded_features = joblib.load('model_features.joblib')

# 模擬使用者輸入數據
# 請確保輸入數據的順序和數量與 loaded_features 匹配
user_input = pd.DataFrame([{
    'TESG分數': 60,
    '環境構面分數': 65,
    '社會構面分數': 70,
    '公司治理構面分數': 55,
    '市值_avg': 1000000000,
    'ROA_lag': 2.5
}])

# 如果你選擇的是 SVM 模型，需要對輸入數據進行標準化
# if isinstance(loaded_model, Pipeline):
#     # 假設 scaler 是 pipeline 中的第一個步驟
#     user_input_scaled = loaded_model.named_steps['scaler'].transform(user_input[loaded_features])
#     prediction = loaded_model.named_steps['svm'].predict(user_input_scaled)
# else:
prediction = loaded_model.predict(user_input[loaded_features])

print(f"預測的未來 ROA: {prediction[0]:.2f}")
```

### Step 25｜模型比較圖

這一步也是沿用舊版 notebook 的做法，但補成三個模型。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

results_summary.plot(x="Model", y="R2", kind="bar", ax=axes[0], legend=False)
axes[0].set_title("R2 Comparison")

results_summary.plot(x="Model", y="RMSE", kind="bar", ax=axes[1], legend=False)
axes[1].set_title("RMSE Comparison")

results_summary.plot(x="Model", y="MAE", kind="bar", ax=axes[2], legend=False)
axes[2].set_title("MAE Comparison")

plt.tight_layout()
plt.show()

### Step 26｜年度平均趨勢圖（報告很適合）

這張圖可以讓你在報告中說：
- ESG 整體趨勢
- ROA 整體趨勢
- 有沒有同向變動

In [ ]:
trend_df = research_df.groupby("年度")[["TESG分數", "ROA_avg"]].mean(numeric_only=True).reset_index()

fig, ax1 = plt.subplots(figsize=(8, 5))

# ESG
ax1.plot(
    trend_df["年度"], trend_df["TESG分數"],
    marker="o", color="blue", label="ESG Score"
)
ax1.set_xlabel("Year")
ax1.set_ylabel("Average ESG Score", color="blue")

# ROA
ax2 = ax1.twinx()
ax2.plot(
    trend_df["年度"], trend_df["ROA_avg"],
    marker="s", color="darkred", label="ROA"
)
ax2.set_ylabel("Average ROA", color="darkred")

plt.title("Average ESG Score and ROA Over Time")

# 合併 legend（關鍵🔥）
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")

plt.tight_layout()
plt.savefig("Fig4_ESG_ROA_trend.png", dpi=300, bbox_inches="tight")
plt.show()

### Step 27｜大公司與小公司平均表現比較

這張圖很適合接異質性分析。

In [ ]:
group_trend = research_df.groupby("Large_Firm")[["TESG分數", "ROA_avg"]].mean(numeric_only=True)
display(group_trend)

group_trend.plot(kind="bar", figsize=(8,5))
plt.title("Average ESG and ROA: Small vs Large Firms")
plt.ylabel("Mean Value")
plt.xticks(rotation=0)
plt.show()

### Step 28｜輸出報告用結果表

把主要結果存成 CSV，之後簡報或報告很好整理。

In [ ]:
results_summary.to_csv("model_results_summary.csv", index=False, encoding="utf-8-sig")
coef_df.to_csv("linear_coefficients.csv", index=False, encoding="utf-8-sig")
rf_importance.to_csv("rf_feature_importance.csv", index=False, encoding="utf-8-sig")

print("已輸出：")
print("- model_results_summary.csv")
print("- linear_coefficients.csv")
print("- rf_feature_importance.csv")

## 預測未來ROA


In [ ]:
# ===== 預測未來 ROA =====

target = "ROA_next"

features = [
    "TESG分數",
    "環境構面分數",
    "社會構面分數",
    "公司治理構面分數",
    "市值_avg",
    "ROA_lag"   # ⭐ 很重要（時間資訊）
]

# 過濾存在欄位
features = [c for c in features if c in research_df.columns]

df_model = research_df.dropna(subset=features + [target]).copy()

X = df_model[features]
y = df_model[target]

print("樣本數:", len(df_model))
print("Features:", features)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression (Predict ROA_next)")
print("R2 :", r2_score(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("MAE :", mean_absolute_error(y_test, y_pred_lr))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest (Predict ROA_next)")
print("R2 :", r2_score(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("MAE :", mean_absolute_error(y_test, y_pred_rf))

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost (Predict ROA_next)")
print("R2 :", r2_score(y_test, y_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
print("MAE :", mean_absolute_error(y_test, y_pred_xgb))

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    random_state=42
)

lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM (Predict ROA_next)")
print("R2  :", r2_score(y_test, y_pred_lgbm))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lgbm)))
print("MAE :", mean_absolute_error(y_test, y_pred_lgbm))

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# SVM對特徵縮放很敏感，所以使用Pipeline來標準化數據
pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVR(kernel='rbf', C=1.0, epsilon=0.1)) # 這裡可以使用不同的核函數和參數
])

pipeline_svm.fit(X_train, y_train)
y_pred_svm = pipeline_svm.predict(X_test)

print("SVM (Predict ROA_next)")
print("R2  :", r2_score(y_test, y_pred_svm))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_svm)))
print("MAE :", mean_absolute_error(y_test, y_pred_svm))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

plt.scatter(y_test, y_pred_xgb, alpha=0.6)

# 完美預測線
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linestyle="--")

plt.xlabel("Actual ROA (t+1)")
plt.ylabel("Predicted ROA (t+1)")
plt.title("Prediction of Future ROA (XGBoost)")

plt.savefig("Fig_Prediction_ROA_next.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 5)) # 調整圖表數量以容納新模型

# Linear
axes[0].scatter(y_test, y_pred_lr, alpha=0.7)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_title("Linear Regression")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")

# RF
axes[1].scatter(y_test, y_pred_rf, alpha=0.7)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[1].set_title("Random Forest")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")

# XGB
axes[2].scatter(y_test, y_pred_xgb, alpha=0.7)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[2].set_title("XGBoost")
axes[2].set_xlabel("Actual")
axes[2].set_ylabel("Predicted")

# LightGBM
if 'y_pred_lgbm' in locals():
    axes[3].scatter(y_test, y_pred_lgbm, alpha=0.7)
    axes[3].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    axes[3].set_title("LightGBM")
    axes[3].set_xlabel("Actual")
    axes[3].set_ylabel("Predicted")
else:
    axes[3].set_title("LightGBM (Not Run)")
    axes[3].text(0.5, 0.5, "Model not run", horizontalalignment='center', verticalalignment='center', transform=axes[3].transAxes, fontsize=12, color='red')
    axes[3].set_xlabel("Actual")
    axes[3].set_ylabel("Predicted")

# SVM
if 'y_pred_svm' in locals():
    axes[4].scatter(y_test, y_pred_svm, alpha=0.7)
    axes[4].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    axes[4].set_title("SVM")
    axes[4].set_xlabel("Actual")
    axes[4].set_ylabel("Predicted")
else:
    axes[4].set_title("SVM (Not Run)")
    axes[4].text(0.5, 0.5, "Model not run", horizontalalignment='center', verticalalignment='center', transform=axes[4].transAxes, fontsize=12, color='red')
    axes[4].set_xlabel("Actual")
    axes[4].set_ylabel("Predicted")


plt.tight_layout()
plt.savefig("vs_all_models.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import joblib
joblib.dump(best_model_to_save, "roa_prediction_model.joblib")
joblib.dump(features, "model_features.pkl")

In [ ]:
from google.colab import files

# 下載模型檔案
try:
    files.download('roa_prediction_model.joblib')
    print("已開始下載 roa_prediction_model.joblib")
except FileNotFoundError:
    print("錯誤：roa_prediction_model.joblib 未找到。請確保模型已成功保存。")

# 下載特徵列表檔案
try:
    files.download('model_features.pkl')
    print("已開始下載 model_features.pkl")
except FileNotFoundError:
    print("錯誤：model_features.pkl 未找到。請確保特徵列表已成功保存。")

In [ ]:
import joblib

joblib.dump(model, "roa_prediction_model.joblib")
joblib.dump(feature_columns, "model_features.pkl")

